# Monte Carlo Value-at-Risk (VaR)

This notebook demonstrates **Monte Carlo Simulation** for estimating portfolio Value-at-Risk (VaR).

Monte Carlo methods generate many simulated paths of portfolio returns, based on estimated mean and covariance from historical data. We then compute the quantile of simulated returns to estimate VaR.

We will:
1. Load a multi-asset portfolio
2. Estimate returns and covariance
3. Run Monte Carlo simulations
4. Compute Monte Carlo VaR
5. Visualize the simulated return distribution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from src.var_methods import monte_carlo_var

## Load Data

We build a portfolio of 4 assets: SPY, BND, GLD, and QQQ.

In [ ]:
tickers = ["SPY", "BND", "GLD", "QQQ"]
data = yf.download(tickers, start="2020-01-01", end="2025-01-01")["Close"]
log_returns = np.log(data / data.shift(1)).dropna()

log_returns.head()

## Monte Carlo VaR

We assume equal portfolio weights and simulate 10,000 paths over a 5-day horizon.

In [ ]:
weights = np.array([0.25, 0.25, 0.25, 0.25])
portfolio_value = 1_000_000
alpha = 0.05

mc_var = monte_carlo_var(
    returns=log_returns,
    weights=weights,
    alpha=alpha,
    horizon=5,
    sims=10000,
    portfolio_value=portfolio_value,
    seed=42
)

print(f"Monte Carlo 95% 5-day VaR: ${mc_var:,.2f}")

## Visualize Simulated Distribution

We plot the histogram of simulated 5-day returns with the 95% VaR cutoff.

In [ ]:
# Simulate raw portfolio returns again for visualization
mu = log_returns.mean().values
cov = log_returns.cov().values

simulated_paths = np.random.multivariate_normal(mu, cov, size=(10000, 5))
simulated_weighted = simulated_paths @ weights
simulated_total = simulated_weighted.sum(axis=1)

plt.figure(figsize=(10,6))
plt.hist(simulated_total, bins=50, alpha=0.7)
plt.axvline(-mc_var/portfolio_value, color="red", linestyle="--", label="95% VaR")
plt.title("Monte Carlo Simulated Portfolio Returns (5-day)")
plt.xlabel("Simulated Return")
plt.ylabel("Frequency")
plt.legend()
plt.show()

## Summary

- Monte Carlo VaR uses simulations of portfolio returns to estimate risk.
- This approach can incorporate correlations, fat tails (with alternative distributions), and complex portfolios.
- It is more flexible than Historical or Parametric VaR, but computationally more expensive.

Next steps (future notebooks):
- Compare Monte Carlo vs. Historical vs. Parametric VaR side by side
- Extend Monte Carlo with **variance reduction** techniques (antithetic variates, control variates)
- Backtest VaR estimates with coverage tests